# Inter annotator agreement for ENTITY

In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

path_to_annotated = 'shared_overlap_entity_annotated.csv'
path_to_gs = '../gs_entity.csv'

df_annotated = pd.read_csv(path_to_annotated)


df_gs = pd.read_csv(path_to_gs)

df_gs['revision_id'] = df_gs['revision_id'].astype(str)
df_gs['property_id'] = df_gs['property_id'].astype(int)
df_gs['value_id'] = df_gs['value_id'].astype(str)

# df["row_id"] = (
#     df["revision_id"].astype(str) + "_" + df["property_id"].astype(str) + "_" + df["value_id"].astype(str)
# )

def split_row_id(row):
    split_ = row.split('_')
    revision_id = split_[0]
    property_id = split_[1]
    value_id = split_[2]
    return revision_id, property_id, value_id

df_annotated.loc[df_annotated.index, ['revision_id', 'property_id', 'value_id']] = df_annotated.apply(
            lambda x: split_row_id(x['row_id']), axis=1, result_type='expand').values

df_annotated['revision_id'] = df_annotated['revision_id'].astype(str)
df_annotated['property_id'] = df_annotated['property_id'].astype(int)
df_annotated['value_id'] = df_annotated['value_id'].astype(str)

df_gs_little = df_gs[['revision_id', 'property_id', 'value_id', 'old_value', 'new_value', 'label']].copy()
df_gs_little.rename(columns={'label': 'my_label'}, inplace=True)
df_merged = df_annotated.merge(df_gs_little, on=['revision_id', 'property_id', 'value_id', 'old_value', 'new_value'], how='left')

df_merged = df_merged.rename(columns={'label': 'other_annotator_label'})
df_merged = df_merged[(~df_merged['my_label'].isna()) & (~df_merged['other_annotator_label'].isna())].copy()

df_merged['my_label'] = df_merged['my_label'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_merged['other_annotator_label'] = df_merged['other_annotator_label'].apply(lambda x: x.strip() if isinstance(x, str) else x)

y1 = df_merged['my_label'].tolist()
y2 = df_merged['other_annotator_label'].tolist()
print(f"Cohen's Kappa: {cohen_kappa_score(y1, y2)}")

agreement = len(df_merged[df_merged['my_label'] == df_merged['other_annotator_label']])
total = len(df_merged)
print('Agreement percentage: ', agreement / total)

print('Number of disagreements:', len(df_merged[df_merged['my_label'] != df_merged['other_annotator_label']]))
display(df_merged[df_merged['my_label'] != df_merged['other_annotator_label']][['old_value_label', 'new_value_label', 'my_label', 'other_annotator_label', 'old_value_description', 'new_value_description']])


Cohen's Kappa: 0.8534733080187625
Agreement percentage:  0.9024390243902439
Number of disagreements: 8


,old_value_label,new_value_label,my_label,other_annotator_label,old_value_description,new_value_description
3,François Valéry,François,property_value_update,unrefinement,French singer,male given name
4,Department of Prints and Drawings of the Louvre,Department of Paintings of the Louvre,property_value_update,refinement,curatorial department of the Louvre,curatorial department of the Louvre in charge ...
10,Fahrenheit: Indigo Prophecy Remastered,Fahrenheit,property_value_update,unrefinement,2015 remaster of the 2005 interactive video ga...,2005 cinematic interactive drama action-advent...
17,;,semicolon,property_value_update,refinement,Unicode character,punctuation mark that separates major sentence...
23,geographic location,geographical feature,property_value_update,unrefinement,"point, line or area on or near Earth",components of planets that can be geographical...
27,Antwerpen-Haven interchange,Antwerp,property_value_update,refinement,interchange in Belgium,"town in Antwerp municipality, Belgium"
73,ethical banking,bank,unrefinement,property_value_update,bank concerned with the social and environment...,financial institution that accepts deposits
87,Downsview Airport,Toronto,unrefinement,property_value_update,airport,capital and largest city of the province of On...


### Cohen Kappa per label for ENTITYT

In [14]:

labels = ['refinement', 'unrefinement', 'property_value_update']
for label in labels:
    # reframes the multi-class problem into a binary one per label
    y1_binary = (df_merged['my_label'] == label).astype(int)  # True for the specific label, False otherwise
    y2_binary = (df_merged['other_annotator_label'] == label).astype(int) # True for the specific label, False otherwise
    print(f"Cohen's Kappa for {label}: {cohen_kappa_score(y1_binary, y2_binary):.2f}")

Cohen's Kappa for refinement: 0.92
Cohen's Kappa for unrefinement: 0.85
Cohen's Kappa for property_value_update: 0.79


# Inter annotator agreement for TEXT

In [15]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from nltk.metrics import masi_distance
from nltk.metrics.agreement import AnnotationTask

def canonicalize(label):
    if pd.isna(label) or not str(label).strip():
        return label
    parts = [p.strip() for p in str(label).split(",")]
    return ", ".join(sorted(parts))

df = pd.read_csv("shared_overlap_text_annotated.csv")
df['my_label'] = df['my_label'].apply(canonicalize)
df['other_annotator_label'] = df['other_annotator_label'].apply(canonicalize)

data = []
for idx, row in df.dropna(subset=['my_label', 'other_annotator_label']).iterrows():
    my_set = frozenset(l.strip() for l in row['my_label'].split(','))
    other_set = frozenset(l.strip() for l in row['other_annotator_label'].split(','))
    data.append(('me', idx, my_set))
    data.append(('other', idx, other_set))

task = AnnotationTask(data=data, distance=masi_distance)
print(f"Krippendorff's Alpha: {task.alpha()}")

agreement = len(df[df['my_label'] == df['other_annotator_label']])
total = len(df)
print('Agreement percentage: ', agreement / total)

print('Number of disagreements:', len(df[df['my_label'] != df['other_annotator_label']]))
display(df[df['my_label'] != df['other_annotator_label']][['old_value', 'new_value', 'my_label', 'other_annotator_label']])

df[df['my_label'] != df['other_annotator_label']][['revision_id', 'value_id', 'property_id', 'old_value', 'new_value', 'my_label', 'other_annotator_label']].to_csv('disagreements.csv', index=False)

Krippendorff's Alpha: 0.7226823019617709
Agreement percentage:  0.775
Number of disagreements: 27


,old_value,new_value,my_label,other_annotator_label
26,"""American politician – Minnesota -Senate 1883-...","""American politician (1840–1908) – Minnesota -...",refinement,"property_value_update, refinement"
27,"""Wu JC""","""Joseph C. Wu""",property_value_update,"property_value_update, refinement"
28,"""University of Nueva Caceres Historical Marker...","""University of Nueva Caceres NHI historical ma...",refinement,"property_value_update, refinement"
29,"""car body configuration with a rear door that ...","""passenger car body in a 2-box configuration w...",refinement,property_value_update
41,"""village in Kilis, Turkey""","""village in Kilis Merkez, Turkey""",refinement,property_value_update
48,"""Choce\u0148 Castle Kitchen garden""","""Set of buildings and greenhouses, Choce\u0148...",unrefinement,refinement
50,"""d. 1854""","""(1803-1854)""",refinement,property_value_update
54,"""Jasenovac concentration camp""","""Jasenovac I concentration camp""",refinement,property_value_update
68,"""Histone cluster 1 H4 family member c""","""Histone cluster 4 H4""",property_value_update,"textual_change, unrefinement"
69,"""legendary ancient Chinese tribal leader and c...","""Legendary ancient Chinese emperor""",property_value_update,unrefinement


## Cohen's Kappa per label

In [16]:
text_labels = ['refinement', 'unrefinement', 'textual_change', 'property_value_update']
text_subset = df.dropna(subset=['my_label', 'other_annotator_label'])

per_label_kappa_text = {}
for label in text_labels:
    y1_binary = text_subset['my_label'].apply(lambda s: label in [x.strip() for x in s.split(',')]).astype(int)
    y2_binary = text_subset['other_annotator_label'].apply(lambda s: label in [x.strip() for x in s.split(',')]).astype(int)
    per_label_kappa_text[label] = cohen_kappa_score(y1_binary, y2_binary)
    print(f"Cohen's Kappa for {label}: {per_label_kappa_text[label]:.2f}")

Cohen's Kappa for refinement: 0.71
Cohen's Kappa for unrefinement: 0.68
Cohen's Kappa for textual_change: 0.84
Cohen's Kappa for property_value_update: 0.63
